# Run Hybrid RAG Inference on GPU

**IMPORTANT: Set Runtime to GPU (T4 or A100)**
- Runtime → Change runtime type → Hardware accelerator: GPU

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Clone repository from GitHub
!git clone https://github.com/matrix-mayank/mathfish.git
%cd mathfish

In [ ]:
# 3. Install dependencies
!pip install -q torch transformers sentence-transformers datasets huggingface-hub tqdm

In [ ]:
# 4. Upload trained models (zip files)
# Upload biencoder_5k_gpu.zip and verifier.zip
from google.colab import files
import zipfile
import os

print("Upload biencoder_5k_gpu.zip:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Extracting {filename}...')
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('outputs/')

print("\nNow upload verifier.zip:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Extracting {filename}...')
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('outputs/')

In [ ]:
# 5. Verify models exist
!ls -lh outputs/biencoder_5k_gpu/checkpoint/
!ls -lh outputs/verifier/

In [ ]:
# 6. Run inference with threshold 0.9 (~2-3 minutes on GPU)
!python scripts/run_hybrid_rag.py \
  --biencoder_path outputs/biencoder_5k_gpu \
  --verifier_path outputs/verifier \
  --problems_file data/processed/problems_dev100.jsonl \
  --output_file outputs/hybrid_dev100_t0.9.jsonl \
  --top_k 20 \
  --verification_threshold 0.9 \
  --device cuda

In [ ]:
# 7. Evaluate results
!python scripts/evaluate_alignment.py \
  --predictions_file outputs/hybrid_dev100_t0.9.jsonl \
  --gold_file data/processed/problems_dev100.jsonl

In [ ]:
# 8. (Optional) Test multiple thresholds
import subprocess
import json

thresholds = [0.5, 0.7, 0.8, 0.85, 0.9, 0.95]
results = []

for t in thresholds:
    print(f"\n===== Testing threshold {t} =====")
    output_file = f"outputs/hybrid_dev100_t{t}.jsonl"
    
    # Run inference
    subprocess.run([
        "python", "scripts/run_hybrid_rag.py",
        "--biencoder_path", "outputs/biencoder_5k_gpu",
        "--verifier_path", "outputs/verifier",
        "--problems_file", "data/processed/problems_dev100.jsonl",
        "--output_file", output_file,
        "--top_k", "20",
        "--verification_threshold", str(t),
        "--device", "cuda"
    ], check=True)
    
    # Evaluate
    result = subprocess.run([
        "python", "scripts/evaluate_alignment.py",
        "--predictions_file", output_file,
        "--gold_file", "data/processed/problems_dev100.jsonl"
    ], capture_output=True, text=True)
    
    metrics = json.loads(result.stdout)
    metrics['threshold'] = t
    results.append(metrics)
    print(f"F1: {metrics['micro_f1']:.4f}, Recall@20: {metrics['recall@20']:.4f}")

# Summary
print("\n===== SUMMARY =====")
print(f"{'Threshold':<12} {'Micro F1':<12} {'Recall@20':<12}")
for r in results:
    print(f"{r['threshold']:<12} {r['micro_f1']:<12.4f} {r['recall@20']:<12.4f}")

In [ ]:
# 9. Download results
import shutil

# Zip all prediction files
shutil.make_archive('hybrid_inference_results', 'zip', 'outputs', '.')

# Download
files.download('hybrid_inference_results.zip')

print("\n✅ Download complete!")